In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import seaborn as sns
import logging

# Import your custom modules
from heston_model import hCommModel
from heston_calibration import hCommCalibrator
from heston_pricer import hCommPricer

# Setup plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

print("✓ All imports successful!")

---
## 1. Data Loading & Visualization

Load the SPY volatility surface and market data, then visualize the implied volatility structure.

In [ ]:
# Load the Excel file
vol_surface = pd.read_excel('SPY_Calibration_Template.xlsx', sheet_name='Vol_Matrix', index_col=0)
market_data = pd.read_excel('SPY_Calibration_Template.xlsx', sheet_name='Market_Data')

print("Volatility Surface Shape:", vol_surface.shape)
print("\nMaturities (years):", vol_surface.index.values[:5], "...")
print("Moneyness levels:", vol_surface.columns.values[:5], "...")
print("\nVolatility Surface (first 5x5):")
print(vol_surface.iloc[:5, :5])

In [ ]:
# Extract key market parameters
S0 = market_data['S0'].iloc[0]  # Spot price
r = market_data['Risk_Free_Rate'].iloc[0]  # Risk-free rate
q = market_data['Div_Yield'].iloc[0]  # Dividend yield (convenience yield for commodities)

print(f"Market Parameters:")
print(f"  Spot Price (S0): ${S0:.2f}")
print(f"  Risk-Free Rate (r): {r:.4f} ({r*100:.2f}%)")
print(f"  Dividend Yield (q): {q:.4f} ({q*100:.2f}%)")

In [ ]:
# Visualize the 3D volatility surface
fig = plt.figure(figsize=(14, 8))
ax = fig.add_subplot(111, projection='3d')

# Create meshgrid
T_grid, K_grid = np.meshgrid(vol_surface.index, vol_surface.columns)
vol_grid = vol_surface.values.T

# Plot surface
surf = ax.plot_surface(T_grid, K_grid, vol_grid, cmap='viridis', alpha=0.8, edgecolor='none')

ax.set_xlabel('Time to Maturity (years)', fontsize=11)
ax.set_ylabel('Moneyness (%)', fontsize=11)
ax.set_zlabel('Implied Volatility', fontsize=11)
ax.set_title('SPY Implied Volatility Surface', fontsize=14, fontweight='bold')

fig.colorbar(surf, ax=ax, shrink=0.5, aspect=5)
plt.tight_layout()
plt.show()

print("✓ 3D volatility surface plotted")

In [ ]:
# Plot volatility smile for different maturities
fig, ax = plt.subplots(figsize=(12, 6))

# Select a few maturities to display
selected_maturities = [0.09, 0.22, 0.47, 0.72, 0.97]  # Approx 1M, 3M, 6M, 9M, 1Y

for T in selected_maturities:
    # Find closest maturity in data
    idx = np.argmin(np.abs(vol_surface.index - T))
    actual_T = vol_surface.index[idx]
    
    vols = vol_surface.iloc[idx]
    ax.plot(vols.index, vols.values, marker='o', label=f'T = {actual_T:.2f}y', linewidth=2)

ax.axvline(x=100, color='red', linestyle='--', alpha=0.5, label='ATM')
ax.set_xlabel('Moneyness (%)', fontsize=12)
ax.set_ylabel('Implied Volatility', fontsize=12)
ax.set_title('Volatility Smile Across Maturities', fontsize=14, fontweight='bold')
ax.legend(loc='best')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("✓ Volatility smiles plotted")

---
## 2. Initialize Heston Model

Create a Heston model with initial parameter guesses. These will be refined during calibration.

In [ ]:
# Initial Heston parameters (reasonable guesses for equities)
initial_params = {
    'kappa': 2.0,      # Mean reversion speed
    'theta': 0.04,     # Long-term variance
    'xi': 0.3,         # Vol of vol
    'rho': -0.7,       # Correlation (negative for equities)
    'v0': 0.04         # Initial variance
}

# Create model
model = hCommModel(initial_params)

print("Initial Heston Parameters:")
for key, val in initial_params.items():
    print(f"  {key}: {val}")

# Check Feller condition
feller_lhs = 2 * model.kappa * model.theta
feller_rhs = model.xi ** 2
print(f"\nFeller Condition Check: 2κθ = {feller_lhs:.4f} {'>' if feller_lhs > feller_rhs else '<'} ξ² = {feller_rhs:.4f}")
print(f"Status: {'✓ Satisfied' if feller_lhs >= feller_rhs else '✗ Violated'}")

---
## 3. Carr-Madan FFT Pricing

Demonstrate fast option pricing using the Fast Fourier Transform method.

In [ ]:
# Price options across a range of strikes using FFT
T_test = 0.5  # 6 months
strikes = np.linspace(S0 * 0.8, S0 * 1.2, 50)  # Strikes from 80% to 120% of spot

# Price using Carr-Madan FFT
fft_prices = model.carr_madan_call_prices(T_test, S0, r, q, strikes, alpha=1.5, N=4096, eta=0.25)

# Price using traditional trapezoidal integration (slower but accurate)
trap_prices = model.h_call_prices_trapezoid(T_test, S0, r, q, strikes[::5])  # Subsample for speed

print(f"Priced {len(strikes)} options using FFT in seconds")
print(f"Sample prices (first 5 strikes):")
for i in range(5):
    print(f"  Strike ${strikes[i]:.2f}: ${fft_prices[i]:.4f}")

In [ ]:
# Plot FFT vs Trapezoidal pricing comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: Price curves
ax1.plot(strikes, fft_prices, 'b-', linewidth=2, label='FFT Pricing')
ax1.plot(strikes[::5], trap_prices, 'ro', markersize=8, label='Trapezoidal (sampled)')
ax1.axvline(x=S0, color='green', linestyle='--', alpha=0.5, label=f'Spot = ${S0:.2f}')
ax1.set_xlabel('Strike Price', fontsize=11)
ax1.set_ylabel('Call Option Price', fontsize=11)
ax1.set_title(f'Heston Call Prices (T={T_test}y)', fontsize=12, fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Right: Pricing errors
errors = np.abs(fft_prices[::5] - trap_prices)
ax2.semilogy(strikes[::5], errors, 'ro-', linewidth=2, markersize=6)
ax2.set_xlabel('Strike Price', fontsize=11)
ax2.set_ylabel('Absolute Pricing Error (log scale)', fontsize=11)
ax2.set_title('FFT vs Trapezoidal: Pricing Accuracy', fontsize=12, fontweight='bold')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nMax pricing error: ${errors.max():.6f}")
print(f"Mean pricing error: ${errors.mean():.6f}")
print("✓ FFT pricing is extremely accurate and much faster!")

---
## 4. Single-Maturity Calibration

Calibrate the Heston model to match market implied volatilities for a single expiration.

In [ ]:
# Create calibrator
calibrator = hCommCalibrator(model)

# Select a maturity to calibrate (e.g., 6 months)
T_calib = 0.47  # Approximately 6 months

# Adjust bounds for equity (vs commodity defaults)
equity_bounds = {
    "kappa": (0.5, 5.0),        # Lower for equities
    "xi": (0.1, 1.5),           # Lower vol of vol
    "rho": (-0.95, -0.3),       # Strongly negative for equities
    "v0": (0.001, 0.5),         # Initial variance
    "theta_scale": (0.1, 2.0)   # Ensures Feller
}

print(f"Starting calibration for T = {T_calib}y...\n")
print("="*60)

In [ ]:
# Run single-T calibration
result_single = calibrator.calibrate_single_T(
    df_vol_surface=vol_surface,
    T=T_calib,
    S0=S0,
    r=r,
    q=q,
    alpha=1.5,
    N_fft=4096,
    eta=0.25,
    bounds=equity_bounds
)

print("\n" + "="*60)
if result_single['success']:
    print("✓ Calibration successful!")
    print(f"\nCalibrated Parameters:")
    for key, val in result_single['params'].items():
        print(f"  {key}: {val:.5f}")
    print(f"\nFinal MSE: {result_single['mse']:.2e}")
    print(f"RMSE: ${result_single['pricing_errors']['rmse']:.4f}")
else:
    print("✗ Calibration failed:", result_single.get('message', 'Unknown error'))

In [ ]:
# Visualize calibration fit
if result_single['success']:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Extract data
    moneyness, sigma_market = calibrator.get_volatility_slice(vol_surface, T_calib)
    strikes = moneyness * S0 / 100
    
    # Market prices
    market_prices = calibrator.bs_call_prices(S0, strikes, T_calib, r, q, sigma_market)
    
    # Model prices with calibrated parameters
    model_prices = model.carr_madan_call_prices(T_calib, S0, r, q, strikes)
    
    # Left: Price comparison
    ax1.plot(moneyness, market_prices, 'bo-', linewidth=2, markersize=8, label='Market Prices')
    ax1.plot(moneyness, model_prices, 'r^--', linewidth=2, markersize=8, label='Heston Model')
    ax1.axvline(x=100, color='green', linestyle='--', alpha=0.5, label='ATM')
    ax1.set_xlabel('Moneyness (%)', fontsize=11)
    ax1.set_ylabel('Call Price', fontsize=11)
    ax1.set_title(f'Calibration Fit: T={T_calib}y', fontsize=12, fontweight='bold')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Right: Percentage errors
    pct_errors = result_single['pricing_errors']['percentage']
    ax2.bar(moneyness, pct_errors, color='coral', alpha=0.7, edgecolor='black')
    ax2.axhline(y=0, color='black', linestyle='-', linewidth=0.8)
    ax2.set_xlabel('Moneyness (%)', fontsize=11)
    ax2.set_ylabel('Pricing Error (%)', fontsize=11)
    ax2.set_title('Model vs Market: Percentage Errors', fontsize=12, fontweight='bold')
    ax2.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.show()
    
    print("✓ Calibration quality visualization complete")

---
## 5. Multi-Maturity Calibration

Calibrate to multiple expiration dates simultaneously to capture the term structure.

In [ ]:
# Reset model to initial guess for multi-T calibration
model = hCommModel(initial_params)
calibrator = hCommCalibrator(model)

# Select multiple maturities (e.g., 3M, 6M, 9M, 1Y)
Ts_multi = [0.22, 0.47, 0.72, 0.97]

# Optional: weights for each maturity (equal by default)
weights = [1.0, 1.5, 1.5, 1.0]  # Give more weight to middle maturities

print(f"Starting multi-T calibration for T = {Ts_multi}...\n")
print("="*60)

In [ ]:
# Run multi-T calibration
result_multi = calibrator.calibrate_multi_T(
    df_vol_surface=vol_surface,
    Ts=Ts_multi,
    S0=S0,
    r=r,
    q=q,
    weights=weights,
    alpha=1.5,
    N_fft=4096,
    eta=0.25
)

print("\n" + "="*60)
if result_multi['success']:
    print("✓ Multi-T calibration successful!")
    print(f"\nCalibrated Parameters:")
    for key, val in result_multi['params'].items():
        print(f"  {key}: {val:.5f}")
    print(f"\nFinal Weighted MSE: {result_multi['mse']:.2e}")
else:
    print("✗ Multi-T calibration failed:", result_multi.get('message', 'Unknown error'))

In [ ]:
# Visualize multi-T calibration fit
if result_multi['success']:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    axes = axes.flatten()
    
    for idx, T in enumerate(Ts_multi):
        ax = axes[idx]
        
        # Extract market data
        moneyness, sigma_market = calibrator.get_volatility_slice(vol_surface, T)
        strikes = moneyness * S0 / 100
        market_prices = calibrator.bs_call_prices(S0, strikes, T, r, q, sigma_market)
        
        # Model prices
        model_prices = model.carr_madan_call_prices(T, S0, r, q, strikes)
        
        # Plot
        ax.plot(moneyness, market_prices, 'bo-', linewidth=2, markersize=6, label='Market')
        ax.plot(moneyness, model_prices, 'r^--', linewidth=2, markersize=6, label='Model')
        ax.axvline(x=100, color='green', linestyle='--', alpha=0.5)
        ax.set_xlabel('Moneyness (%)', fontsize=10)
        ax.set_ylabel('Call Price', fontsize=10)
        ax.set_title(f'T = {T:.2f}y', fontsize=11, fontweight='bold')
        ax.legend()
        ax.grid(True, alpha=0.3)
    
    fig.suptitle('Multi-Maturity Calibration Results', fontsize=14, fontweight='bold', y=1.00)
    plt.tight_layout()
    plt.show()
    
    print("✓ Multi-T calibration quality visualization complete")

---
## 6. Monte Carlo Simulation

Generate price paths using the calibrated Heston model parameters.

In [ ]:
# Create simulation configuration
from heston_model import hCommModel

# Note: The simConfig is defined inside hCommModel, so we need to create it properly
# For now, we'll use keyword arguments directly in a dict-like structure

sim_params = {
    'S0': S0,           # Initial spot
    'T': 1.0,           # 1 year simulation
    'r': r,             # Risk-free rate
    'q': q,             # Convenience yield
    'M': 5000,          # Number of paths
    'N': 252,           # Daily steps (252 trading days)
    'seed': 42,         # For reproducibility
    'alpha': 0.0,       # No mean reversion overlay
    'mu': 0.0,
    'A': 0.0,           # No seasonality
    'phi': 0.0
}

print("Simulation Configuration:")
print(f"  Paths: {sim_params['M']:,}")
print(f"  Time Steps: {sim_params['N']}")
print(f"  Maturity: {sim_params['T']} years")
print(f"\nUsing calibrated Heston parameters from multi-T calibration...")

In [ ]:
# Note: The simulate method signature may need adjustment based on your actual implementation
# This is a workaround since the dataclass isn't decorated properly

# Run simulation (adjust based on your actual simulate() signature)
try:
    # Try using the config object if it works
    config = type('SimConfig', (), sim_params)()
    spot_paths, var_paths = model.simulate(config)
except:
    # Fallback: use keyword arguments directly
    print("Using fallback simulation method...")
    # You may need to adjust this based on your actual implementation
    spot_paths, var_paths = model.simulate(
        S0=sim_params['S0'],
        T=sim_params['T'],
        r=sim_params['r'],
        q=sim_params['q'],
        npaths=sim_params['M'],
        nsteps=sim_params['N'],
        seed=sim_params['seed']
    )

print(f"\n✓ Simulation complete!")
print(f"  Spot paths shape: {spot_paths.shape}")
print(f"  Variance paths shape: {var_paths.shape}")
print(f"\nTerminal Statistics:")
print(f"  Mean: ${spot_paths[-1].mean():.2f}")
print(f"  Std Dev: ${spot_paths[-1].std():.2f}")
print(f"  Min: ${spot_paths[-1].min():.2f}")
print(f"  Max: ${spot_paths[-1].max():.2f}")

In [ ]:
# Plot simulated paths
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10))

# Time grid
time_grid = np.linspace(0, sim_params['T'], sim_params['N'] + 1)

# Top: Spot price paths
n_display = 100  # Display 100 paths for clarity
for i in range(n_display):
    ax1.plot(time_grid, spot_paths[:, i], alpha=0.3, linewidth=0.5, color='steelblue')

# Add mean path
mean_path = spot_paths.mean(axis=1)
ax1.plot(time_grid, mean_path, 'r-', linewidth=3, label='Mean Path')
ax1.axhline(y=S0, color='green', linestyle='--', linewidth=2, label=f'Initial Spot = ${S0:.2f}')

ax1.set_xlabel('Time (years)', fontsize=11)
ax1.set_ylabel('Spot Price', fontsize=11)
ax1.set_title(f'Heston Model: {n_display} Simulated Price Paths', fontsize=12, fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Bottom: Variance paths
for i in range(n_display):
    ax2.plot(time_grid, var_paths[:, i], alpha=0.3, linewidth=0.5, color='coral')

mean_var = var_paths.mean(axis=1)
ax2.plot(time_grid, mean_var, 'darkred', linewidth=3, label='Mean Variance')
ax2.axhline(y=model.theta, color='blue', linestyle='--', linewidth=2, label=f'Long-run θ = {model.theta:.4f}')

ax2.set_xlabel('Time (years)', fontsize=11)
ax2.set_ylabel('Variance', fontsize=11)
ax2.set_title(f'Variance Paths (CIR Process)', fontsize=12, fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("✓ Path simulation visualized")

In [ ]:
# Terminal distribution analysis
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: Histogram
ax1.hist(spot_paths[-1], bins=50, density=True, alpha=0.7, color='steelblue', edgecolor='black')
ax1.axvline(x=S0, color='green', linestyle='--', linewidth=2, label=f'Initial = ${S0:.2f}')
ax1.axvline(x=spot_paths[-1].mean(), color='red', linestyle='-', linewidth=2, label=f'Mean = ${spot_paths[-1].mean():.2f}')
ax1.set_xlabel('Terminal Spot Price', fontsize=11)
ax1.set_ylabel('Density', fontsize=11)
ax1.set_title('Distribution of Terminal Prices', fontsize=12, fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Right: Q-Q plot to check normality of log-returns
from scipy.stats import probplot
log_returns = np.log(spot_paths[-1] / S0)
probplot(log_returns, dist="norm", plot=ax2)
ax2.set_title('Q-Q Plot: Log-Returns vs Normal', fontsize=12, fontweight='bold')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nLog-Return Statistics:")
print(f"  Mean: {log_returns.mean():.4f}")
print(f"  Std Dev: {log_returns.std():.4f}")
print(f"  Skewness: {pd.Series(log_returns).skew():.4f}")
print(f"  Kurtosis: {pd.Series(log_returns).kurtosis():.4f}")

---
## 7. Exotic Options Pricing

Use the calibrated model to price various exotic options.

In [ ]:
# Initialize pricer
pricer = hCommPricer(model)

# Common parameters
T_exotic = 0.5  # 6 months
K_exotic = S0   # ATM strike

print("Exotic Options Pricing Demo")
print("="*60)
print(f"Underlying: ${S0:.2f}")
print(f"Strike: ${K_exotic:.2f}")
print(f"Maturity: {T_exotic} years")
print(f"Risk-free rate: {r:.4f}")
print(f"Dividend yield: {q:.4f}")
print("="*60)

In [ ]:
# 1. European Vanilla Options (analytical)
print("\n1. EUROPEAN VANILLA OPTIONS (Analytical Heston)")
call_price = pricer.price_european(T_exotic, S0, r, q, K_exotic, ot='call')
put_price = pricer.price_european(T_exotic, S0, r, q, K_exotic, ot='put')

print(f"   Call: ${call_price:.4f}")
print(f"   Put:  ${put_price:.4f}")

# Verify put-call parity
parity_check = call_price - put_price - (S0 * np.exp(-q * T_exotic) - K_exotic * np.exp(-r * T_exotic))
print(f"   Put-Call Parity Check: {abs(parity_check):.2e} (should be ~0)")

In [ ]:
# 2. Digital (Binary) Options
print("\n2. DIGITAL OPTIONS (Monte Carlo)")
digital_call = pricer.price_digital(T_exotic, S0, r, q, K_exotic, ot='call', payout=100.0, n_paths=100000, seed=42)
digital_put = pricer.price_digital(T_exotic, S0, r, q, K_exotic, ot='put', payout=100.0, n_paths=100000, seed=42)

print(f"   Digital Call (pays $100 if S_T > K): ${digital_call:.4f}")
print(f"   Digital Put (pays $100 if S_T < K):  ${digital_put:.4f}")

In [ ]:
# 3. Barrier Options
print("\n3. BARRIER OPTIONS (Monte Carlo)")

barrier_up = S0 * 1.15    # 15% above spot
barrier_down = S0 * 0.85  # 15% below spot

# Up-and-Out Call
uao_call = pricer.price_barrier(T_exotic, S0, r, q, K_exotic, barrier_up, 
                                barrier_type='UpAndOut', ot='call', n_paths=100000, n_steps=252, seed=42)
print(f"   Up-and-Out Call (barrier=${barrier_up:.2f}): ${uao_call:.4f}")

# Down-and-Out Call
dao_call = pricer.price_barrier(T_exotic, S0, r, q, K_exotic, barrier_down, 
                                barrier_type='DownAndOut', ot='call', n_paths=100000, n_steps=252, seed=42)
print(f"   Down-and-Out Call (barrier=${barrier_down:.2f}): ${dao_call:.4f}")

# Up-and-In Put
uai_put = pricer.price_barrier(T_exotic, S0, r, q, K_exotic, barrier_up, 
                               barrier_type='UpAndIn', ot='put', n_paths=100000, n_steps=252, seed=42)
print(f"   Up-and-In Put (barrier=${barrier_up:.2f}): ${uai_put:.4f}")

print(f"\n   Note: Vanilla Call ≈ Up-and-Out + Up-and-In")
# This relationship should approximately hold (but we priced call analytically, barrier via MC)

In [ ]:
# 4. Options on Futures
print("\n4. OPTIONS ON FUTURES")
futures_price = S0 * np.exp((r - q) * T_exotic)  # Futures price
futures_call = pricer.price_on_futures(T_exotic, futures_price, r, K_exotic, ot='call', n_paths=100000, seed=42)

print(f"   Futures Price: ${futures_price:.2f}")
print(f"   Call on Futures: ${futures_call:.4f}")

In [ ]:
# 5. Options with Mean Reversion (commodity feature)
print("\n5. MEAN-REVERTING OPTIONS (Commodity Feature)")
mr_speed = 0.5   # Mean reversion speed
mr_level = S0    # Revert to current spot

mr_call = pricer.price_with_mean_reversion(T_exotic, S0, r, q, K_exotic, 
                                           mr_speed, mr_level, ot='call', n_paths=100000, seed=42)
print(f"   Call with Mean Reversion (α={mr_speed}, μ=${mr_level:.2f}): ${mr_call:.4f}")
print(f"   Vanilla Call (no MR): ${call_price:.4f}")
print(f"   Difference: ${abs(mr_call - call_price):.4f}")

In [ ]:
# 6. Options with Seasonality (commodity feature)
print("\n6. SEASONAL OPTIONS (Commodity Feature)")
seasonal_amp = 0.05  # 5% seasonal swing
seasonal_phase = 0.25  # Phase shift

seasonal_call = pricer.price_with_seasonality(T_exotic, S0, r, q, K_exotic, 
                                              seasonal_amp, seasonal_phase, ot='call', n_paths=100000, seed=42)
print(f"   Call with Seasonality (A={seasonal_amp}, φ={seasonal_phase}): ${seasonal_call:.4f}")
print(f"   Vanilla Call (no seasonality): ${call_price:.4f}")
print(f"   Difference: ${abs(seasonal_call - call_price):.4f}")

---
## 8. Summary & Comparison

Final summary of all results from this demonstration.

In [ ]:
# Create summary table
summary_data = {
    'Algorithm': [
        'Carr-Madan FFT',
        'Single-T Calibration',
        'Multi-T Calibration',
        'Monte Carlo Simulation',
        'European Vanilla',
        'Digital Option',
        'Barrier Option',
        'Futures Option',
        'Mean Reversion',
        'Seasonality'
    ],
    'Status': [
        '✓ Demonstrated',
        '✓ Calibrated' if result_single['success'] else '✗ Failed',
        '✓ Calibrated' if result_multi['success'] else '✗ Failed',
        '✓ Simulated',
        '✓ Priced',
        '✓ Priced',
        '✓ Priced',
        '✓ Priced',
        '✓ Priced',
        '✓ Priced'
    ],
    'Key Result': [
        f'{len(strikes)} strikes priced',
        f"MSE: {result_single['mse']:.2e}" if result_single['success'] else 'N/A',
        f"MSE: {result_multi['mse']:.2e}" if result_multi['success'] else 'N/A',
        f"{sim_params['M']:,} paths",
        f"Call: ${call_price:.2f}",
        f"${digital_call:.2f}",
        f"${uao_call:.2f}",
        f"${futures_call:.2f}",
        f"${mr_call:.2f}",
        f"${seasonal_call:.2f}"
    ]
}

summary_df = pd.DataFrame(summary_data)

print("\n" + "="*80)
print("COMPLETE HESTON MODEL DEMONSTRATION SUMMARY")
print("="*80)
print(summary_df.to_string(index=False))
print("="*80)

print("\nCalibrated Heston Parameters (Multi-T):")
if result_multi['success']:
    for key, val in result_multi['params'].items():
        print(f"  {key}: {val:.5f}")

print("\n✓ All demonstrations complete!")
print("\nNext steps:")
print("  1. Experiment with different calibration targets")
print("  2. Try commodity datasets (energy, metals, agriculture)")
print("  3. Implement Greeks calculation and hedging strategies")
print("  4. Explore different parameter bounds for various asset classes")